In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
import shap
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, recall_score, f1_score, precision_score, average_precision_score
import warnings
warnings.filterwarnings('ignore')

from config import *

In [3]:
# Load processed data (split files)
X_train = pd.read_csv(PROCESSED_DIR / 'X_train.csv')
X_test  = pd.read_csv(PROCESSED_DIR / 'X_test.csv')
y_train = pd.read_csv(PROCESSED_DIR / 'y_train.csv').squeeze()
y_test  = pd.read_csv(PROCESSED_DIR / 'y_test.csv').squeeze()

# Combine for full dataset (Optuna ke liye)
X = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
y = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)

print("✅ Data Loaded Successfully!")
print("Shape:", X.shape)
print("Target distribution:\n", y.value_counts(normalize=True))
print("Columns count:", len(X.columns))

✅ Data Loaded Successfully!
Shape: (101763, 35)
Target distribution:
 readmitted_binary
0    0.888398
1    0.111602
Name: proportion, dtype: float64
Columns count: 35


In [4]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from config import *

# ====================== LOAD DATA ======================
X_train = pd.read_csv(PROCESSED_DIR / 'X_train.csv')
X_test  = pd.read_csv(PROCESSED_DIR / 'X_test.csv')
y_train = pd.read_csv(PROCESSED_DIR / 'y_train.csv').squeeze()
y_test  = pd.read_csv(PROCESSED_DIR / 'y_test.csv').squeeze()

X = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
y = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)

print("✅ Data Loaded Successfully!")
print("Total Shape:", X.shape)
print("\nTarget Distribution:")
print(y.value_counts(normalize=True))

✅ Data Loaded Successfully!
Total Shape: (101763, 35)

Target Distribution:
readmitted_binary
0    0.888398
1    0.111602
Name: proportion, dtype: float64


In [5]:
import optuna
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

def objective(trial):
    param = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 31, 128),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 80),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 6.0, 12.0),
        'random_state': RANDOM_STATE,
        'verbose': -1
    }
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(**param)
        model.fit(X_tr, y_tr, 
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(60, verbose=False)])
        
        pred_proba = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, pred_proba))
    
    return np.mean(scores)


# ====================== RUN OPTUNA ======================
print("🚀 Starting Optuna Hyperparameter Tuning...")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=40)   # 40 trials (15-30 min lag sakte hain)

print("\n✅ Best AUC:", study.best_value)
print("Best Params:", study.best_params)

[I 2026-06-04 23:30:08,014] A new study created in memory with name: no-name-6c232214-a042-4f5f-962c-501c1327d78f


🚀 Starting Optuna Hyperparameter Tuning...


[I 2026-06-04 23:30:23,143] Trial 0 finished with value: 0.6766101366380834 and parameters: {'num_leaves': 32, 'max_depth': 8, 'learning_rate': 0.014483069819797814, 'n_estimators': 1333, 'min_child_samples': 46, 'subsample': 0.766749716066278, 'colsample_bytree': 0.7864671319496945, 'reg_alpha': 0.3144771929775969, 'reg_lambda': 2.024558975887319, 'scale_pos_weight': 8.889867466590118}. Best is trial 0 with value: 0.6766101366380834.
[I 2026-06-04 23:30:40,171] Trial 1 finished with value: 0.6762980638238244 and parameters: {'num_leaves': 77, 'max_depth': 5, 'learning_rate': 0.02865683553706424, 'n_estimators': 962, 'min_child_samples': 64, 'subsample': 0.9504225301184237, 'colsample_bytree': 0.7903548833481835, 'reg_alpha': 3.746911149316043, 'reg_lambda': 1.4092122796342128, 'scale_pos_weight': 9.6019941009364}. Best is trial 0 with value: 0.6766101366380834.
[I 2026-06-04 23:30:45,465] Trial 2 finished with value: 0.6744636400398876 and parameters: {'num_leaves': 125, 'max_depth': 


✅ Best AUC: 0.6775545617319867
Best Params: {'num_leaves': 64, 'max_depth': 12, 'learning_rate': 0.038873370732288984, 'n_estimators': 1091, 'min_child_samples': 58, 'subsample': 0.9196892788076453, 'colsample_bytree': 0.733969566445622, 'reg_alpha': 2.971381262864414, 'reg_lambda': 1.5859597253908335, 'scale_pos_weight': 6.424494841527111}


In [6]:
# Best Parameters
best_params = study.best_params
best_params.update({
    'objective': 'binary',
    'metric': 'auc',
    'random_state': RANDOM_STATE,
    'verbose': -1
})

# Final Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

# Train Final Model
final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X_train, y_train)

# Predictions
y_pred_proba = final_model.predict_proba(X_test)[:, 1]
optimal_threshold = 0.15

y_pred = (y_pred_proba >= optimal_threshold).astype(int)

# Results
from sklearn.metrics import recall_score, f1_score, average_precision_score

print("\n" + "="*50)
print("FINAL MODEL PERFORMANCE")
print("="*50)
print(f"ROC-AUC          : {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"PR-AUC           : {average_precision_score(y_test, y_pred_proba):.4f}")
print(f"Recall @0.15     : {recall_score(y_test, y_pred):.4f}")
print(f"F1 @0.15         : {f1_score(y_test, y_pred):.4f}")
print(f"Optimal Threshold: {optimal_threshold}")


FINAL MODEL PERFORMANCE
ROC-AUC          : 0.6605
PR-AUC           : 0.2067
Recall @0.15     : 0.9335
F1 @0.15         : 0.2184
Optimal Threshold: 0.15


In [7]:
# Save Final Model
joblib.dump(final_model, MODELS_DIR / 'best_readmission_model.pkl')
print(f"✅ Best Model Saved at: {MODELS_DIR / 'best_readmission_model.pkl'}")

# Save Best Parameters
joblib.dump(best_params, MODELS_DIR / 'best_params.pkl')

✅ Best Model Saved at: d:\Projects\healthcare-readmission-predictor\notebooks\..\models\best_readmission_model.pkl


['d:\\Projects\\healthcare-readmission-predictor\\notebooks\\..\\models\\best_params.pkl']